# SC Research Notebook

Layer 1 (INE crude) data pulls for the China aromatics value chain.

**Scripts:** `code/sc_term_structure.py` · `code/sc_basis.py` · `code/china_crude_supply_demand.py` · `code/fetch_refinery_utilization.py` · `code/sc_warehouse_receipts.py` · `code/px_spot.py` · `code/ta_eg_basis.py`  
**Wind (Dubai daily):** `code/fetch_wind.py` — requires Wind terminal + WindPy  
**Data docs:** `data/sc_data_sources.md`

Run all cells to refresh CSV/JSON outputs in `data/`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CODE = ROOT / "code"
DATA = ROOT / "data"
sys.path.insert(0, str(CODE))

print(f"Project root: {ROOT}")
print(f"Data dir: {DATA}")

In [ ]:
# 1) SC term structure snapshot (front vs ~6M)
from sc_term_structure import build_term_structure_snapshot

curve, term_summary = build_term_structure_snapshot()
display(curve.head(10))
print(json.dumps(term_summary, indent=2))

In [ ]:
# 2) SC vs Brent/Dubai basis (Dubai = Wind EDB daily when terminal available)
from sc_basis import build_basis_panel, summarize_basis

basis_panel, dubai_source = build_basis_panel()
basis_summary = summarize_basis(basis_panel, dubai_source=dubai_source)
display(basis_panel.tail())
print(f"Dubai source: {dubai_source}")
print(json.dumps(basis_summary, indent=2)) 

In [ ]:
# 3) China crude imports vs refining + CDU utilization
from china_crude_supply_demand import build_supply_demand_panel, summarize
from fetch_refinery_utilization import load_refinery_utilization, summarize_refinery_utilization

supply_panel = build_supply_demand_panel()
util_df = load_refinery_utilization()
util_summary = summarize_refinery_utilization(util_df)
supply_summary = summarize(supply_panel, util_summary)
display(supply_panel.tail())
display(util_df)
print(json.dumps(supply_summary, indent=2))

In [ ]:
# Persist all outputs (same as running CLI scripts)
import sc_term_structure
import sc_basis
import china_crude_supply_demand
import fetch_refinery_utilization
import sc_warehouse_receipts

sc_term_structure.main()
sc_basis.main()
china_crude_supply_demand.main()
fetch_refinery_utilization.main()
sc_warehouse_receipts.main()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 2, figsize=(11, 4))

# plot the term structure
c = curve.sort_values("contract_month")
axes[0,0].plot(c["contract_month"], c["settle"], marker="o")
axes[0,0].set_title("SC curve (settle, CNY/bbl)")
axes[0,0].tick_params(axis="x", rotation=45)

# plot the volume
axes[1,0].bar(c["contract_month"], c["volume"], width=1, color="steelblue", alpha=0.7)
axes[1,0].set_ylabel("Volume")
axes[1,0].tick_params(axis="x", rotation=45)

# plot the basis
bp = basis_panel.set_index("date")
axes[0,1].plot(bp.index, bp["basis_sc_minus_brent_usd"], lw=0.8)
axes[0,1].axhline(0, color="k", lw=0.5)
axes[0,1].set_title("SC − Brent basis (USD/bbl)")
axes[0,1].tick_params(axis="x", rotation=45)

#plot the crude balance
sp = supply_panel.dropna(subset=["crude_imports_tbpd", "refinery_runs_tbpd"])
axes[1,1].plot(sp["period"], sp["crude_imports_tbpd"], label="imports")
axes[1,1].plot(sp["period"], sp["refinery_runs_tbpd"], label="refinery runs")
axes[1,1].set_title("China crude balance (Tb/d)")
axes[1,1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4) INE SC warehouse receipts + fitted models
import numpy as np
from sc_warehouse_receipts import build_receipt_panel

receipt_df, receipt_summary = build_receipt_panel()
display(receipt_df.tail(10))
print(json.dumps(receipt_summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)

ax0 = axes[0]
ax0.plot(receipt_df["date"], receipt_df["receipt"] / 1e4, lw=0.9, color="darkorange")
ax0.set_ylabel("Receipts (10k bbl)")
ax0.set_title("INE SC warehouse receipts")
ax0.grid(alpha=0.3)

models = receipt_summary["models"]
t = np.arange(len(receipt_df))
intercept = models["linear_trend"]["intercept_bbl"]
slope = models["linear_trend"]["slope_bbl_per_day"]
ax0.plot(
    receipt_df["date"],
    (intercept + slope * t) / 1e4,
    "--",
    color="gray",
    lw=1,
    label=f"OLS trend ({slope:.0f} bbl/day)",
)
ax0.legend()

ax1 = axes[1]
ax1.bar(receipt_df["date"], receipt_df["receipt_chg"], width=1, color="steelblue", alpha=0.6)
ax1.axhline(0, color="k", lw=0.5)
ax1.set_ylabel("Daily change (bbl)")
ax1.set_title("Receipt change")
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.show()